In [1]:
import numpy as np


In [2]:
import numpy as np

class ACO():
    def __init__(self, deposit_type="global", Q=1.0):
        self.deposit_type = deposit_type
        self.Q = Q
        
        if self.deposit_type == "global":
            self.deposit = self._global_deposit
        elif self.deposit_type == "local":
            self.deposit = self._local_deposit
        elif self.deposit_type == "constant":
            self.deposit = self._constant_deposit

    def run(self, D, n_ants, rho, alpha, beta=0.0, max_iters=1000):
        self.n = D.shape[0]
        self.n_ants = n_ants
        self.D = D 
  
        with np.errstate(divide='ignore'):
            self.eta = np.where(D > 0, 1.0 / D, 0.0)

        self.tau = self._initialize()

        best_road = None
        best_dist = float('inf')

        for t in range(max_iters):
            all_roads = []
            all_dist = []
    
            for k in range(self.n_ants):
                road = [0] 
                visited = {0}

                while len(road) < self.n:
                    current_node = road[-1]
                    next_node = self._generate_next_node(current_node, visited, alpha, beta)
                    road.append(next_node)
                    visited.add(next_node)
                    
                road.append(road[0])  
                all_roads.append(road)
                
    
                dist_k = self._calculate_road_distance(road)
                all_dist.append(dist_k)

   
                if dist_k < best_dist:
                    best_dist = dist_k
                    best_road = road.copy()


            self._evaporate(rho)

   
            self.deposit(all_roads, all_dist)

        return best_road, best_dist

    def _initialize(self):
        """Inicializa la matriz tau con valores pequeños y positivos"""
        return np.random.uniform(0.1, 0.5, size=(self.n, self.n))
    
    def _evaporate(self, rho):
        """Aplica la regla de evaporación (1 - rho) * tau"""
        self.tau = (1 - rho) * self.tau

    def _calculate_road_distance(self, road):
        """Calcula la longitud total de un camino cerrado"""
        distance = 0.0
        for i in range(len(road) - 1):
            distance += self.D[road[i], road[i+1]]
        return distance

    def _generate_next_node(self, current_node, visited, alpha, beta):
        """Selecciona el siguiente nodo usando la regla de transición probabilística"""
        unvisited = [node for node in range(self.n) if node not in visited]
        
        if not unvisited:
            return 0  
        
        probabilities = []
        for j in unvisited:
            tau_val = self.tau[current_node, j] ** alpha
        
            eta_val = (self.eta[current_node, j] ** beta) if beta > 0 else 1.0
            probabilities.append(tau_val * eta_val)
        
        probabilities = np.array(probabilities)
        sum_prob = np.sum(probabilities)
        
        if sum_prob == 0:
            probabilities = np.ones(len(unvisited)) / len(unvisited)
        else:
            probabilities /= sum_prob

        return np.random.choice(unvisited, p=probabilities)

    def _global_deposit(self, all_roads, all_dist):
        """Variante Ant-Cycle: El depósito depende del costo total del viaje"""
        for k in range(self.n_ants):
            road_k = all_roads[k]
            dist_k = all_dist[k]
            
            delta_tau_k = self.Q / dist_k 

            for i in range(len(road_k) - 1):
                u, v = road_k[i], road_k[i+1]
                self.tau[u, v] += delta_tau_k
                self.tau[v, u] += delta_tau_k  

    def _local_deposit(self, all_roads, all_dist):
        """Variante Ant-Quantity: El depósito depende de la distancia de cada arco individual"""
        for k in range(self.n_ants):
            road_k = all_roads[k]
            
            for i in range(len(road_k) - 1):
                u, v = road_k[i], road_k[i+1]
                dist_arco = self.D[u, v]
                
                if dist_arco > 0:
                    delta_tau_k = self.Q / dist_arco
                    self.tau[u, v] += delta_tau_k
                    self.tau[v, u] += delta_tau_k

    def _constant_deposit(self, all_roads, all_dist):
        """Variante Ant-Density: Se deposita un valor fijo constante en cada arco usado"""
        for k in range(self.n_ants):
            road_k = all_roads[k]
            
            for i in range(len(road_k) - 1):
                u, v = road_k[i], road_k[i+1]
                self.tau[u, v] += self.Q
                self.tau[v, u] += self.Q